# Data Inspection

#### Locating dataset files

In [14]:
from pathlib import Path
import pandas as pd
import soundfile as sf

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw" / "training_data"

wav_files = sorted(DATA_DIR.rglob("*.wav"))
txt_files = sorted(DATA_DIR.rglob("*.txt"))
tsv_files = sorted(DATA_DIR.rglob("*.tsv"))

print("Audio recordings:", len(wav_files))
print("Text files:", len(txt_files))
print("Annotation files:", len(tsv_files))

if not wav_files:
    raise FileNotFoundError("No WAV files found. Check DATA_DIR.")

Audio recordings: 3163
Text files: 942
Annotation files: 3163


#### Patient Inspection
- Inspecting the contents of one patient's files directly

In [15]:
patient_files = [
    path for path in txt_files
    if list(path.parent.glob(f"{path.stem}_*.wav"))
]

if not patient_files:
    raise FileNotFoundError("No patient text files with matching audio found.")

example_patient = patient_files[0]

print("patient file:", example_patient.name)
print(example_patient.read_text())

patient file: 13918.txt
13918 4 4000
AV 13918_AV.hea 13918_AV.wav 13918_AV.tsv
PV 13918_PV.hea 13918_PV.wav 13918_PV.tsv
TV 13918_TV.hea 13918_TV.wav 13918_TV.tsv
MV 13918_MV.hea 13918_MV.wav 13918_MV.tsv
#Age: Child
#Sex: Male
#Height: 98.0
#Weight: 15.9
#Pregnancy status: False
#Murmur: Present
#Murmur locations: TV
#Most audible location: TV
#Systolic murmur timing: Holosystolic
#Systolic murmur shape: Plateau
#Systolic murmur grading: I/VI
#Systolic murmur pitch: Low
#Systolic murmur quality: Blowing
#Diastolic murmur timing: nan
#Diastolic murmur shape: nan
#Diastolic murmur grading: nan
#Diastolic murmur pitch: nan
#Diastolic murmur quality: nan
#Outcome: Abnormal
#Campaign: CC2015
#Additional ID: nan



- Note that a `murmur_label` of `Present` only applies to a specific `murmur_location`, not everywhere.

#### Creating the recordings index
- This reads each patient file, extracts its murmur label, and collects information about its audio recordings

In [16]:
recordings = pd.read_csv(
    OUTPUT_DIR / "recording_index.csv",
    dtype={"patient_id": str}
)

patients = pd.read_csv(
    OUTPUT_DIR / "patient_index.csv",
    dtype={"patient_id": str}
)

#### Inspecting the dataset
- Note that we're counting subjects separately from recordings; someone with five recordings must count as one patient, not five

In [17]:
patients = recordings[
    ["patient_id", "murmur_label"]
].drop_duplicates()

print("Patients:", patients["patient_id"].nunique())
print("Recordings:", len(recordings))

print("\nPatients per class:")
print(patients["murmur_label"].value_counts(dropna=False))

print("\nRecordings per class:")
print(recordings["murmur_label"].value_counts(dropna=False))

print("\nSample rates:")
print(recordings["sample_rate"].value_counts())

print("\nRecording locations:")
print(recordings["location"].value_counts())

print("\nRecording duration summary:")
print(recordings["duration_seconds"].describe())

print("\nRecordings per patient:")
print(recordings.groupby("patient_id").size().describe())

print("\nFile or label issues:")
display(pd.DataFrame(issues))

Patients: 942
Recordings: 3163

Patients per class:
murmur_label
Absent     695
Present    179
Unknown     68
Name: count, dtype: int64

Recordings per class:
murmur_label
Absent     2391
Present     616
Unknown     156
Name: count, dtype: int64

Sample rates:
sample_rate
4000    3163
Name: count, dtype: int64

Recording locations:
location
MV     861
AV     800
PV     766
TV     732
Phc      4
Name: count, dtype: int64

Recording duration summary:
count    3163.000000
mean       22.870301
std         7.282948
min         5.152000
25%        19.056000
50%        21.456000
75%        29.392000
max        64.512000
Name: duration_seconds, dtype: float64

Recordings per patient:
count    942.000000
mean       3.357749
std        0.998638
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max        6.000000
dtype: float64

File or label issues:


""


#### Splitting the dataset
- Using a 70% , 15%, 15% split for training / validation / test
- Splits are done at the *patient-level*, so all files of a patient are bundled together in the same split

In [20]:
patient_splits = pd.read_csv(
    OUTPUT_DIR / "patient_splits.csv",
    dtype={"patient_id": str}
)

recording_splits = pd.read_csv(
    OUTPUT_DIR / "recording_splits.csv",
    dtype={"patient_id": str}
)

display(pd.crosstab(
    patient_splits["split"],
    patient_splits["murmur_label"]
))

display(recording_splits.head())

murmur_label,Absent,Present,Unknown
split,,,
test,105,27,10
train,486,125,48
validation,104,27,10


,patient_id,recording,location,murmur_label,sample_rate,duration_seconds,channels,audio_path,split
0,13918,13918_AV.wav,AV,Present,4000,10.288,1,data/raw/training_data/13918_AV.wav,test
1,13918,13918_PV.wav,PV,Present,4000,9.392,1,data/raw/training_data/13918_PV.wav,test
2,13918,13918_TV.wav,TV,Present,4000,6.592,1,data/raw/training_data/13918_TV.wav,test
3,13918,13918_MV.wav,MV,Present,4000,28.784,1,data/raw/training_data/13918_MV.wav,test
4,14241,14241_AV.wav,AV,Present,4000,20.480,1,data/raw/training_data/14241_AV.wav,train
